In [1]:
import pandas as pd

In [2]:
import pandas as pd
df = pd.read_csv("train.csv")
print(df.columns.tolist())
df.head()

['project_id', 'num_commits', 'active_commit_days', 'project_span_days', 'commit_cadence', 'pct_commits_weekend', 'pct_commits_late_night', 'lines_added_total', 'lines_deleted_total', 'churn_ratio', 'median_lines_per_commit', 'max_lines_in_one_commit', 'pct_code_in_first_commit', 'num_commit_authors', 'avg_commit_msg_length', 'pct_generic_commit_msgs', 'num_branches', 'num_merge_commits', 'lines_of_code', 'num_source_files', 'num_directories', 'max_dir_depth', 'num_languages', 'primary_language', 'pct_primary_language', 'num_dependencies', 'framework', 'uses_game_engine', 'has_tests', 'num_test_files', 'has_ci_config', 'has_dockerfile', 'has_backend', 'has_database', 'avg_file_length', 'comment_line_ratio', 'num_image_files', 'num_audio_files', 'num_3d_model_files', 'num_font_files', 'asset_size_mb', 'has_custom_art', 'readme_word_count', 'readme_num_headings', 'readme_num_images', 'readme_num_code_blocks', 'readme_has_install_instructions', 'has_license', 'has_docs_folder', 'url_is_li

,project_id,num_commits,active_commit_days,project_span_days,commit_cadence,pct_commits_weekend,pct_commits_late_night,lines_added_total,lines_deleted_total,churn_ratio,...,forks,num_issues,num_pull_requests,num_releases,is_forked_from_template,author_prior_projects,author_account_age_years,hours_spent,log_hours,hours_bucket
0,535ce026f7,44,15,56,daily,0.2045,0.4545,8500,2900,0.3438,...,0,0,0,0,False,0,1-2,32.2,3.4726,30-100
1,869ba41ed7,9,3,76,sporadic,0.8889,0.0000,500,100,0.2484,...,0,0,0,0,False,0,3-5,5.0,1.6094,<10
2,d6f81f8862,12,1,0,frequent,1.0000,0.0000,5900,100,0.0247,...,0,0,0,0,False,0,3-5,1.3,0.2624,<10
3,152b5d915d,4,1,0,frequent,0.0000,0.2500,6000,0,0.0000,...,0,0,0,0,False,0,<1,2.0,0.6931,<10
4,3b5f8a49e6,7,3,64,sporadic,0.8571,0.0000,800,0,0.0419,...,0,0,0,0,False,0,1-2,4.0,1.3863,<10


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_log_error
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
train = pd.read_csv("train.csv")

DROP = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
features = [c for c in train.columns if c not in DROP]
X = train[features].copy()
for col in features:
    if not pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].astype("category")

X_tr, X_va, y_tr, y_va, h_tr, h_va = train_test_split(
    X, train["log_hours"], train["hours_spent"], test_size=0.2, random_state=42
)
m = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, categorical_features="from_dtype", random_state=42)
m.fit(X_tr, y_tr)
preds = np.expm1(m.predict(X_va)).clip(0.1, 140.7)
print("Local RMSLE:", root_mean_squared_log_error(h_va, preds))

Local RMSLE: 0.7506365303506434


In [33]:
print("Columns:", train.columns.tolist())
print("\nData types:\n", train.dtypes)
print("\nFirst 3 rows:\n", train.head(3))

Columns: ['project_id', 'num_commits', 'active_commit_days', 'project_span_days', 'commit_cadence', 'pct_commits_weekend', 'pct_commits_late_night', 'lines_added_total', 'lines_deleted_total', 'churn_ratio', 'median_lines_per_commit', 'max_lines_in_one_commit', 'pct_code_in_first_commit', 'num_commit_authors', 'avg_commit_msg_length', 'pct_generic_commit_msgs', 'num_branches', 'num_merge_commits', 'lines_of_code', 'num_source_files', 'num_directories', 'max_dir_depth', 'num_languages', 'primary_language', 'pct_primary_language', 'num_dependencies', 'framework', 'uses_game_engine', 'has_tests', 'num_test_files', 'has_ci_config', 'has_dockerfile', 'has_backend', 'has_database', 'avg_file_length', 'comment_line_ratio', 'num_image_files', 'num_audio_files', 'num_3d_model_files', 'num_font_files', 'asset_size_mb', 'has_custom_art', 'readme_word_count', 'readme_num_headings', 'readme_num_images', 'readme_num_code_blocks', 'readme_has_install_instructions', 'has_license', 'has_docs_folder', '

In [36]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_log_error
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

DROP = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
train = pd.read_csv("train.csv")


def add_features(d):
    d = d.copy()
    d["commits_per_day"] = d.num_commits / (d.active_commit_days + 1)
    d["lines_per_commit"] = d.lines_added_total / (d.num_commits + 1)
    d["lines_per_active_day"] = d.lines_added_total / (d.active_commit_days + 1)
    d["loc_per_file"] = d.lines_of_code / (d.num_source_files + 1)
    d["log_lines_added"] = np.log1p(d.lines_added_total)
    d["log_commits"] = np.log1p(d.num_commits)
    d["span_per_active"] = d.project_span_days / (d.active_commit_days + 1)
    d["asset_total"] = (d.num_image_files + d.num_audio_files
                        + d.num_3d_model_files + d.num_font_files)
    d["churn_lines"] = d.lines_added_total + d.lines_deleted_total
    return d


train = add_features(train)
features = [c for c in train.columns if c not in DROP]

cat_cols = []
for c in features:
    if train[c].dtype == bool:
        train[c] = train[c].astype(int)
    elif not pd.api.types.is_numeric_dtype(train[c]):
        cat_cols.append(c)
        cats = sorted(train[c].dropna().astype(str).unique())
        train[c] = pd.Categorical(train[c].astype(str).where(train[c].notna()), categories=cats)

y = train["log_hours"].values
h_true = train["hours_spent"].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_hgb = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_et = np.zeros(len(train))

X_ohe = pd.get_dummies(train[features], columns=cat_cols, dtype=float).fillna(-1)

for tr_idx, va_idx in kf.split(train):
    X_tr, y_tr = train.iloc[tr_idx][features], y[tr_idx]
    X_va, y_va = train.iloc[va_idx][features], y[va_idx]

    hgb = HistGradientBoostingRegressor(
        max_iter=900, learning_rate=0.015, max_leaf_nodes=6,
        min_samples_leaf=60, l2_regularization=10.0, max_features=0.5,
        categorical_features="from_dtype", random_state=42,
    )
    hgb.fit(X_tr, y_tr)
    oof_hgb[va_idx] = hgb.predict(X_va)

    if HAS_LGB:
        model_lgb = lgb.LGBMRegressor(
            n_estimators=1200, learning_rate=0.015, num_leaves=7,
            min_child_samples=55, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.5, reg_alpha=2.0, reg_lambda=10.0,
            random_state=42, verbose=-1,
        )
        model_lgb.fit(X_tr, y_tr)
        oof_lgb[va_idx] = model_lgb.predict(X_va)

    et = ExtraTreesRegressor(
        n_estimators=500, min_samples_leaf=5, max_features=0.5,
        n_jobs=-1, random_state=42,
    )
    et.fit(X_ohe.iloc[tr_idx], y_tr)
    oof_et[va_idx] = et.predict(X_ohe.iloc[va_idx])

print("HGB 5-Fold CV RMSLE:", round(root_mean_squared_log_error(h_true, np.expm1(oof_hgb).clip(0.1, 140.7)), 6))
if HAS_LGB:
    print("LGB 5-Fold CV RMSLE:", round(root_mean_squared_log_error(h_true, np.expm1(oof_lgb).clip(0.1, 140.7)), 6))
print("ET  5-Fold CV RMSLE:", round(root_mean_squared_log_error(h_true, np.expm1(oof_et).clip(0.1, 140.7)), 6))

if HAS_LGB:
    blend = 0.35 * oof_hgb + 0.40 * oof_lgb + 0.25 * oof_et
else:
    blend = 0.50 * oof_hgb + 0.50 * oof_et

print("Blend 5-Fold CV RMSLE:", round(root_mean_squared_log_error(h_true, np.expm1(blend).clip(0.1, 140.7)), 6))


HGB 5-Fold CV RMSLE: 0.723332
LGB 5-Fold CV RMSLE: 0.722494
ET  5-Fold CV RMSLE: 0.729642
Blend 5-Fold CV RMSLE: 0.715093


In [3]:
# Pre-ship check: runs the real predict.py on 500 rows it never trained on
import os, shutil, subprocess, sys, tempfile
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_log_error

full = pd.read_csv("train.csv")
hold = full.sample(500, random_state=11)
rest = full.drop(hold.index)

tmp = tempfile.mkdtemp()
rest.to_csv(os.path.join(tmp, "train.csv"), index=False)
hold.drop(columns=["hours_spent", "log_hours", "hours_bucket"]).to_csv(os.path.join(tmp, "test.csv"), index=False)
shutil.copy("predict.py", tmp)

r = subprocess.run([sys.executable, "predict.py"], cwd=tmp, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print("CRASHED:\n", r.stderr[-1500:])
else:
    sub = pd.read_csv(os.path.join(tmp, "predictions.csv"))
    assert len(sub) == len(hold) and sub.hours_spent.notna().all(), "bad predictions.csv"
    p = sub.set_index("project_id").loc[hold.project_id, "hours_spent"].values
    print("Rows:", len(sub), "| min/max hours:", sub.hours_spent.min(), sub.hours_spent.max())
    print("Holdout RMSLE (500 unseen rows):", round(root_mean_squared_log_error(hold.hours_spent.values, p), 4))
shutil.rmtree(tmp)

hgb: done (3 seeds, 3s)
lgb: done (3 seeds, 7s)
cat skipped (seed 42): No module named 'catboost'
et: done (3 seeds, 9s)
wrote predictions.csv with 500 rows

Rows: 500 | min/max hours: 1.06 37.9
Holdout RMSLE (500 unseen rows): 0.6267


In [3]:
import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
import predict

train = pd.read_csv("train.csv")
y = np.log1p(train["hours_spent"].values)
h_true = train["hours_spent"].values

X_tr, _, cat_cols = predict.prepare_data(train, train.head(5))
X_tr_xgb = X_tr.copy()
for c in cat_cols:
    X_tr_xgb[c] = X_tr_xgb[c].astype("category")

oof_cat = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (tr_i, va_i) in enumerate(kf.split(X_tr)):
    cb = CatBoostRegressor(
        iterations=2000, learning_rate=0.025, depth=6, l2_leaf_reg=5.0,
        cat_features=cat_cols, random_seed=42 + fold, verbose=0, allow_writing_files=False,
    )
    cb.fit(X_tr.iloc[tr_i], y[tr_i], eval_set=(X_tr.iloc[va_i], y[va_i]), early_stopping_rounds=100)
    oof_cat[va_i] = cb.predict(X_tr.iloc[va_i])

    xm = XGBRegressor(
        n_estimators=2000, learning_rate=0.02, max_depth=4, min_child_weight=20,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=1.0, reg_lambda=5.0,
        enable_categorical=True, tree_method="hist", early_stopping_rounds=100,
        random_state=42 + fold, n_jobs=-1, verbosity=0,
    )
    xm.fit(X_tr_xgb.iloc[tr_i], y[tr_i], eval_set=[(X_tr_xgb.iloc[va_i], y[va_i])], verbose=False)
    oof_xgb[va_i] = xm.predict(X_tr_xgb.iloc[va_i])

print("CatBoost 5-Fold CV:", round(root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(oof_cat))), 6))
print("XGBoost  5-Fold CV:", round(root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(oof_xgb))), 6))
print("Blend    5-Fold CV:", round(root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(0.75 * oof_cat + 0.25 * oof_xgb))), 6))

# Dry-run predict.py end-to-end
train.head(50).drop(columns=["hours_spent", "log_hours", "hours_bucket"]).to_csv("test.csv", index=False)
predict.main()
sub = pd.read_csv("predictions.csv")
print("Nulls:", sub.isna().sum().to_dict(), "| Sample:", sub["hours_spent"].head(5).round(2).tolist())
os.remove("test.csv")
os.remove("predictions.csv")


CatBoost 5-Fold CV: 0.666429
XGBoost  5-Fold CV: 0.661937
Blend    5-Fold CV: 0.662312
Nulls: {'project_id': 0, 'hours_spent': 0} | Sample: [24.36, 3.53, 3.03, 4.79, 4.2]


In [4]:
import pandas as pd
train = pd.read_csv("train.csv")
print("Rows, Cols:", train.shape)
print("hours_spent quantiles:\n", train["hours_spent"].quantile([0, 0.05, 0.25, 0.5, 0.75, 0.95, 1.0]))
print("\nFirst 5 project_ids:", train["project_id"].head().tolist())
print("Last 5 project_ids:", train["project_id"].tail().tolist())
print("\nCategory distribution:\n", train["project_category"].value_counts(dropna=False))

Rows, Cols: (4500, 73)
hours_spent quantiles:
 0.00      0.10
0.05      1.00
0.25      2.70
0.50      6.00
0.75     12.90
0.95     44.01
1.00    140.70
Name: hours_spent, dtype: float64

First 5 project_ids: ['535ce026f7', '869ba41ed7', 'd6f81f8862', '152b5d915d', '3b5f8a49e6']
Last 5 project_ids: ['dc213c11fd', 'e9e50b7461', '043d6ec699', 'a7c266c77c', '251d9a19c7']

Category distribution:
 project_category
game       1687
app         961
website     939
tool        697
bot         128
other        83
NaN           5
Name: count, dtype: int64


In [5]:
import warnings
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
import lightgbm as lgb
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import KFold, StratifiedKFold
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

train = pd.read_csv("train.csv")
DROP = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
y = np.log1p(train["hours_spent"].values)
h_true = train["hours_spent"].values

print("=== 1. DATASET DIAGNOSTICS ===")
print("Shape:", train.shape)
print("Hours quantiles:\n", train["hours_spent"].quantile([0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0]).to_dict())
print("Missing cols top 10:\n", train.isna().sum().sort_values(ascending=False).head(10).to_dict())

# Check if row order has drift (time-ordered train.csv vs hidden future test.csv!)
n = len(train)
first_half_mean = y[: n // 2].mean()
second_half_mean = y[n // 2 :].mean()
print(f"First-half log_hours mean: {first_half_mean:.4f} | Second-half mean: {second_half_mean:.4f}")


def prep_raw(df):
    feats = [c for c in df.columns if c not in DROP]
    X = df[feats].copy()
    cat_cols = []
    for c in feats:
        if X[c].dtype == bool or str(X[c].dtype) == "boolean":
            X[c] = X[c].astype("float32")
        elif not pd.api.types.is_numeric_dtype(X[c]):
            cat_cols.append(c)
            fill_v = "other" if c == "project_category" else ""
            X[c] = X[c].fillna(fill_v).astype(str)
        else:
            X[c] = pd.to_numeric(X[c], errors="coerce").astype("float32")
    return X, cat_cols


def prep_fe(df):
    d = df.copy()
    d["commits_per_day"] = d.num_commits / (d.active_commit_days + 1)
    d["lines_per_commit"] = d.lines_added_total / (d.num_commits + 1)
    d["lines_per_active_day"] = d.lines_added_total / (d.active_commit_days + 1)
    d["loc_per_file"] = d.lines_of_code / (d.num_source_files + 1)
    d["active_ratio"] = d.active_commit_days / (d.project_span_days + 1)
    d["churn_lines"] = d.lines_added_total + d.lines_deleted_total
    d["net_loc_ratio"] = d.lines_of_code / (d.lines_added_total + 1)
    d["asset_total"] = d.num_image_files + d.num_audio_files + d.num_3d_model_files + d.num_font_files
    return prep_raw(d)


def eval_config(name, X, cat_cols, cv_splits, model_fn, use_cat_dtype=False):
    oof = np.zeros(len(X))
    Xm = X.copy()
    if use_cat_dtype:
        for c in cat_cols:
            Xm[c] = Xm[c].astype("category")
    for fold, (tr_i, va_i) in enumerate(cv_splits):
        oof[va_i] = model_fn(Xm.iloc[tr_i], y[tr_i], Xm.iloc[va_i], y[va_i], cat_cols, 42 + fold)
    score = root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(oof)))
    print(f"  {name:<35} -> RMSLE: {score:.6f}")
    return oof, score


# Compare StratifiedKFold (on hours_bucket) vs Sequential KFold (no shuffle, tests time drift)
skf = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(train, train["hours_bucket"].fillna("unk")))
seq_kf = list(KFold(n_splits=5, shuffle=False).split(train))

X_raw, cats_raw = prep_raw(train)
X_fe, cats_fe = prep_fe(train)

print("\n=== 2. MODEL & REGULARIZATION SWEEP (5-FOLD STRATIFIED) ===")
oofs = {}

# CatBoost configs (no early stopping vs early stopping, depth 4 vs 5 vs 6)
oofs["cb_d4_reg10"] = eval_config(
    "CatBoost depth=4 l2=10 (1600 it)", X_raw, cats_raw, skf,
    lambda Xa, ya, Xb, yb, cc, s: CatBoostRegressor(
        iterations=1600, learning_rate=0.02, depth=4, l2_leaf_reg=10.0,
        cat_features=cc, random_seed=s, verbose=0, allow_writing_files=False
    ).fit(Xa, ya).predict(Xb)
)[0]

oofs["cb_d5_reg12"] = eval_config(
    "CatBoost depth=5 l2=12 (1600 it)", X_raw, cats_raw, skf,
    lambda Xa, ya, Xb, yb, cc, s: CatBoostRegressor(
        iterations=1600, learning_rate=0.02, depth=5, l2_leaf_reg=12.0,
        cat_features=cc, random_seed=s, verbose=0, allow_writing_files=False
    ).fit(Xa, ya).predict(Xb)
)[0]

oofs["cb_d5_fe"] = eval_config(
    "CatBoost depth=5 l2=12 + Ratios", X_fe, cats_fe, skf,
    lambda Xa, ya, Xb, yb, cc, s: CatBoostRegressor(
        iterations=1600, learning_rate=0.02, depth=5, l2_leaf_reg=12.0,
        cat_features=cc, random_seed=s, verbose=0, allow_writing_files=False
    ).fit(Xa, ya).predict(Xb)
)[0]

oofs["xgb_d3_reg10"] = eval_config(
    "XGBoost depth=3 l2=10 (1200 it)", X_raw, cats_raw, skf,
    lambda Xa, ya, Xb, yb, cc, s: XGBRegressor(
        n_estimators=1200, learning_rate=0.015, max_depth=3, min_child_weight=40,
        subsample=0.8, colsample_bytree=0.5, reg_alpha=2.0, reg_lambda=10.0,
        enable_categorical=True, tree_method="hist", random_state=s, n_jobs=-1, verbosity=0
    ).fit(Xa, ya).predict(Xb),
    use_cat_dtype=True
)[0]

oofs["lgb_l7_reg10"] = eval_config(
    "LightGBM leaves=7 l2=10 (1200 it)", X_raw, cats_raw, skf,
    lambda Xa, ya, Xb, yb, cc, s: lgb.LGBMRegressor(
        n_estimators=1200, learning_rate=0.015, num_leaves=7, min_child_samples=50,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_alpha=2.0, reg_lambda=10.0,
        random_state=s, verbose=-1
    ).fit(Xa, ya).predict(Xb),
    use_cat_dtype=True
)[0]

print("\n=== 3. SEQUENTIAL (NO-SHUFFLE) CHECK FOR TIME DRIFT ===")
eval_config(
    "CatBoost depth=5 (No-Shuffle CV)", X_raw, cats_raw, seq_kf,
    lambda Xa, ya, Xb, yb, cc, s: CatBoostRegressor(
        iterations=1600, learning_rate=0.02, depth=5, l2_leaf_reg=12.0,
        cat_features=cc, random_seed=s, verbose=0, allow_writing_files=False
    ).fit(Xa, ya).predict(Xb)
)

print("\n=== 4. OPTIMAL BLEND WEIGHTS ===")
names = list(oofs.keys())
M = np.column_stack([oofs[k] for k in names])

def loss_fn(w):
    w = np.array(w) / np.sum(w)
    pred = np.maximum(0.05, np.expm1(M @ w))
    return root_mean_squared_log_error(h_true, pred)

res = minimize(loss_fn, np.ones(len(names)) / len(names), bounds=[(0, 1)] * len(names))
best_w = res.x / np.sum(res.x)
for n_m, w_m in zip(names, best_w):
    print(f"  Weight {n_m:<20}: {w_m:.3f}")
print(f"Optimal Blend CV RMSLE: {res.fun:.6f}")


=== 1. DATASET DIAGNOSTICS ===
Shape: (4500, 73)
Hours quantiles:
 {0.0: 0.1, 0.01: 0.5, 0.05: 1.0, 0.25: 2.7, 0.5: 6.0, 0.75: 12.9, 0.95: 44.01000000000003, 0.99: 109.00100000000002, 1.0: 140.7}
Missing cols top 10:
 {'is_mobile_responsive': 1423, 'num_js_files_loaded': 1423, 'num_internal_pages': 1423, 'page_weight_kb': 1349, 'load_time_ms': 1349, 'comment_line_ratio': 194, 'avg_file_length': 194, 'pct_primary_language': 194, 'pct_code_in_first_commit': 186, 'churn_ratio': 21}
First-half log_hours mean: 2.0453 | Second-half mean: 2.0305

=== 2. MODEL & REGULARIZATION SWEEP (5-FOLD STRATIFIED) ===
  CatBoost depth=4 l2=10 (1600 it)    -> RMSLE: 0.674161
  CatBoost depth=5 l2=12 (1600 it)    -> RMSLE: 0.670550
  CatBoost depth=5 l2=12 + Ratios     -> RMSLE: 0.669829
  XGBoost depth=3 l2=10 (1200 it)     -> RMSLE: 0.665366
  LightGBM leaves=7 l2=10 (1200 it)   -> RMSLE: 0.667443

=== 3. SEQUENTIAL (NO-SHUFFLE) CHECK FOR TIME DRIFT ===
  CatBoost depth=5 (No-Shuffle CV)    -> RMSLE: 0.66

In [6]:
import os
import warnings
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import StratifiedKFold
import predict

warnings.filterwarnings("ignore")

train = pd.read_csv("train.csv")
y = np.log1p(train["hours_spent"].values)
h_true = train["hours_spent"].values
strata = train["hours_bucket"].fillna("unk").astype(str)

X_tr, _, cat_cols = predict.prepare_data(train, train.head(5))
skf = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_tr, strata))

configs = [
    ("d5_l2_10 (current predict.py)", dict(iterations=1500, learning_rate=0.02, depth=5, l2_leaf_reg=10.0, random_strength=2.0, subsample=0.8, bootstrap_type="Bernoulli")),
    ("d6_l2_3  (CatBoost default)",   dict(iterations=2000, learning_rate=0.02, depth=6, l2_leaf_reg=3.0)),
    ("d6_l2_5  (Ship #4 style)",      dict(iterations=2000, learning_rate=0.02, depth=6, l2_leaf_reg=5.0, subsample=0.8, bootstrap_type="Bernoulli")),
    ("d7_l2_4  (deeper trees)",       dict(iterations=2000, learning_rate=0.015, depth=7, l2_leaf_reg=4.0, subsample=0.8, bootstrap_type="Bernoulli")),
    ("d8_l2_6  (max interaction)",    dict(iterations=2000, learning_rate=0.015, depth=8, l2_leaf_reg=6.0, subsample=0.8, bootstrap_type="Bernoulli")),
]

best_name, best_score, best_oof = None, 999.0, None
oofs = {}

for name, params in configs:
    oof = np.zeros(len(X_tr))
    for fold, (tr_i, va_i) in enumerate(skf):
        cb = CatBoostRegressor(
            **params,
            cat_features=cat_cols,
            random_seed=42 + fold,
            verbose=0,
            allow_writing_files=False,
        )
        cb.fit(X_tr.iloc[tr_i], y[tr_i])
        oof[va_i] = cb.predict(X_tr.iloc[va_i])
    score = root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(oof)))
    oofs[name] = oof
    print(f"{name:<32} -> 5-Fold RMSLE: {score:.6f}")
    if score < best_score:
        best_name, best_score, best_oof = name, score, oof

# Check if blending top depths (d6 + d7) beats single depth
blend_67 = 0.5 * oofs["d6_l2_5  (Ship #4 style)"] + 0.5 * oofs["d7_l2_4  (deeper trees)"]
print(f"{'Blend d6 + d7':<32} -> 5-Fold RMSLE: {root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(blend_67))):.6f}")

# Dry-run predict.py to confirm zero errors
train.head(50).drop(columns=["hours_spent", "log_hours", "hours_bucket"]).to_csv("test.csv", index=False)
predict.main()
sub = pd.read_csv("predictions.csv")
print("Dry-run nulls:", sub.isna().sum().to_dict(), "| Sample:", sub["hours_spent"].head(5).round(2).tolist())
os.remove("test.csv")
os.remove("predictions.csv")


d5_l2_10 (current predict.py)    -> 5-Fold RMSLE: 0.668746
d6_l2_3  (CatBoost default)      -> 5-Fold RMSLE: 0.666223
d6_l2_5  (Ship #4 style)         -> 5-Fold RMSLE: 0.663752
d7_l2_4  (deeper trees)          -> 5-Fold RMSLE: 0.662475
d8_l2_6  (max interaction)       -> 5-Fold RMSLE: 0.661899
Blend d6 + d7                    -> 5-Fold RMSLE: 0.662339
Dry-run nulls: {'project_id': 0, 'hours_spent': 0} | Sample: [24.36, 3.53, 3.03, 4.79, 4.2]


In [ ]:
import os
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import StratifiedKFold
from scipy.optimize import minimize

os.makedirs("models", exist_ok=True)

train = pd.read_csv("train.csv")
DROP = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
features = [c for c in train.columns if c not in DROP]

y = np.log1p(train["hours_spent"].values)
h_true = train["hours_spent"].values
strata = train["hours_bucket"].fillna("unk").astype(str)

# Prepare CatBoost matrix (exact #1 preprocessing)
X_cb = train[features].copy()
cat_cols = []
for c in features:
    if X_cb[c].dtype == bool or str(X_cb[c].dtype) == "boolean":
        X_cb[c] = X_cb[c].astype("float32")
    elif not pd.api.types.is_numeric_dtype(X_cb[c]):
        cat_cols.append(c)
        fill_v = "other" if c == "project_category" else ""
        X_cb[c] = X_cb[c].fillna(fill_v).astype(str)
    else:
        X_cb[c] = pd.to_numeric(X_cb[c], errors="coerce").astype("float32")

# Prepare XGBoost matrix (exact #1 preprocessing: nullable boolean + NaN-preserving category)
X_xgb = train[features].copy()
bool_cols = [c for c in features if X_xgb[c].dtype == bool or str(X_xgb[c].dtype) == "boolean"]
for c in bool_cols:
    X_xgb[c] = X_xgb[c].astype("boolean")
xgb_cats = {}
for c in cat_cols:
    X_xgb[c] = X_xgb[c].astype("category")
    xgb_cats[c] = list(X_xgb[c].cat.categories)

SEEDS = [42, 7, 2026]
oof_cb7 = np.zeros(len(train))
oof_cb8 = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

idx = 0
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for fold, (tr_i, va_i) in enumerate(skf.split(X_cb, strata)):
        cb7 = CatBoostRegressor(
            iterations=2200, learning_rate=0.015, depth=7, l2_leaf_reg=4.0,
            subsample=0.8, bootstrap_type="Bernoulli", cat_features=cat_cols,
            random_seed=seed + fold, verbose=0, allow_writing_files=False
        ).fit(X_cb.iloc[tr_i], y[tr_i])
        cb7.save_model(f"models/cb7_{idx}.cbm")
        oof_cb7[va_i] += cb7.predict(X_cb.iloc[va_i]) / len(SEEDS)

        cb8 = CatBoostRegressor(
            iterations=2200, learning_rate=0.015, depth=8, l2_leaf_reg=6.0,
            subsample=0.8, bootstrap_type="Bernoulli", cat_features=cat_cols,
            random_seed=seed + fold + 100, verbose=0, allow_writing_files=False
        ).fit(X_cb.iloc[tr_i], y[tr_i])
        cb8.save_model(f"models/cb8_{idx}.cbm")
        oof_cb8[va_i] += cb8.predict(X_cb.iloc[va_i]) / len(SEEDS)

        xm = XGBRegressor(
            n_estimators=1600, learning_rate=0.012, max_depth=4, min_child_weight=30,
            subsample=0.8, colsample_bytree=0.5, reg_alpha=1.5, reg_lambda=8.0,
            enable_categorical=True, tree_method="hist",
            random_state=seed + fold, n_jobs=-1, verbosity=0
        ).fit(X_xgb.iloc[tr_i], y[tr_i])
        xm.save_model(f"models/xgb_{idx}.json")
        oof_xgb[va_i] += xm.predict(X_xgb.iloc[va_i]) / len(SEEDS)

        idx += 1
        print(f"Trained fold {idx}/15")

# Find optimal out-of-fold weights across the 3 model families
M = np.column_stack([oof_cb7, oof_cb8, oof_xgb])
res = minimize(
    lambda w: root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(M @ (w / np.sum(w))))),
    [0.45, 0.45, 0.10], bounds=[(0, 1)] * 3
)
weights = (res.x / np.sum(res.x)).tolist()
print("Optimal weights [cb7, cb8, xgb]:", [round(w, 4) for w in weights])
print("3-Seed Out-of-Fold CV RMSLE:", round(res.fun, 6))

with open("models/meta.json", "w") as f:
    json.dump({
        "features": features,
        "cat_cols": cat_cols,
        "bool_cols": bool_cols,
        "xgb_cats": xgb_cats,
        "n_models": idx,
        "weights": weights,
    }, f)
print("Saved all 45 models + models/meta.json successfully!")
